In [ ]:
"""
Stock Research Report Analyzer
================================
Analyzes quarterly research reports (MOSL, AMBIT, AXIS, PL) for a given stock ticker
across multiple quarters and generates longitudinal insights using Claude AI.

Requirements:
    pip install pypdf pdfplumber anthropic rich

Usage:
    python stock_research_analyzer.py --ticker RELIANCE --quarters 10
    python stock_research_analyzer.py --ticker HDFCBANK --quarters 8 --output html

Configuration:
    Edit the CONFIG section below to match your folder path and API key.
"""

import os
import re
import json
import argparse
import sys
from pathlib import Path
from datetime import datetime
from typing import Optional
import textwrap

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — Edit these to match your setup
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Your folder where all quarter subfolders live
    "base_folder": r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\ResearchReports\CompanyResearchReports",

    # Your Anthropic API key — get free at https://console.anthropic.com
    # Free tier gives ~$5 credit. Each analysis costs ~$0.05–0.15 depending on report length.
    # Alternatively set environment variable: ANTHROPIC_API_KEY=your_key_here
    "anthropic_api_key": "key",  # Leave blank to use env var

    # Model to use (claude-haiku-3-5 is cheapest; claude-sonnet-4-5 is smarter)
    "model": "claude-haiku-4-5-20251001",

    # Max pages to extract from each PDF (reduces token cost; 15 is usually enough)
    "max_pages_per_pdf": 20,

    # Output folder for reports (defaults to same as base_folder)
    "output_folder": "",  # Leave blank to use base_folder
}

# Known broker suffixes in filenames
BROKER_MAP = {
    "MOSL": "Motilal Oswal",
    "AMBIT": "Ambit Capital",
    "AXIS": "Axis Securities",
    "PL": "Prabhudas Liladhar",
    "KOTAK": "Kotak Securities",
}

# Quarter folder pattern: FY26Q3, FY25Q1, etc.
QUARTER_PATTERN = re.compile(r"^FY(\d{2})Q([1-4])$", re.IGNORECASE)

# File pattern: FY26Q3_RELIANCE_MOSL.pdf or FY26Q3_RELIANCE_MOSL_1.pdf
FILE_PATTERN = re.compile(
    r"^(FY\d{2}Q[1-4])_([A-Z0-9]+)_([A-Z]+)(?:_(\d+))?\.pdf$", re.IGNORECASE
)


# ─────────────────────────────────────────────────────────────────────────────
# Utility: Pretty console output
# ─────────────────────────────────────────────────────────────────────────────
try:
    from rich.console import Console
    from rich.progress import track
    from rich.panel import Panel
    from rich.table import Table
    console = Console()
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    class Console:
        def print(self, *args, **kwargs): print(*args)
        def rule(self, *args, **kwargs): print("─" * 60)
    console = Console()
    def track(iterable, description=""):
        print(f"{description}...")
        return iterable
    def Panel(text, **kw): return text


def log(msg, style=""):
    if HAS_RICH:
        console.print(msg, style=style)
    else:
        print(msg)


# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Discover reports for a ticker
# ─────────────────────────────────────────────────────────────────────────────
def quarter_sort_key(qfolder: str) -> tuple:
    """Returns (fiscal_year, quarter_num) for sorting. FY26Q3 → (26, 3)"""
    m = QUARTER_PATTERN.match(qfolder)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    return (0, 0)


def discover_reports(base_folder: str, ticker: str, max_quarters: int = 12) -> list[dict]:
    """
    Walks the base folder, finds all quarter subfolders, and collects
    PDF files matching the ticker. Returns sorted list (newest first).
    """
    base = Path(base_folder)
    if not base.exists():
        log(f"[red]ERROR: Base folder not found: {base_folder}[/red]")
        sys.exit(1)

    # Find all quarter folders
    quarter_folders = [
        d for d in base.iterdir()
        if d.is_dir() and QUARTER_PATTERN.match(d.name)
    ]
    quarter_folders.sort(key=lambda d: quarter_sort_key(d.name), reverse=True)

    log(f"\n[cyan]Found {len(quarter_folders)} quarter folders.[/cyan]")

    reports = []
    quarters_with_data = 0

    for qfolder in quarter_folders:
        if quarters_with_data >= max_quarters:
            break

        quarter_reports = []
        for pdf_file in sorted(qfolder.glob("*.pdf")):
            m = FILE_PATTERN.match(pdf_file.name)
            if not m:
                continue

            file_quarter = m.group(1).upper()
            file_ticker = m.group(2).upper()
            file_broker = m.group(3).upper()
            file_version = m.group(4)  # Could be None

            if file_ticker != ticker.upper():
                continue

            quarter_reports.append({
                "quarter": file_quarter,
                "ticker": file_ticker,
                "broker": file_broker,
                "broker_name": BROKER_MAP.get(file_broker, file_broker),
                "version": file_version or "0",
                "path": str(pdf_file),
                "filename": pdf_file.name,
            })

        if quarter_reports:
            reports.extend(quarter_reports)
            quarters_with_data += 1
            log(f"  [green]✓[/green] {qfolder.name}: {len(quarter_reports)} report(s) found")
        else:
            log(f"  [dim]✗ {qfolder.name}: no reports for {ticker}[/dim]")

    return reports


# ─────────────────────────────────────────────────────────────────────────────
# Step 2: Extract text from PDFs
# ─────────────────────────────────────────────────────────────────────────────
def extract_pdf_text(pdf_path: str, max_pages: int = 20) -> str:
    """
    Extracts text from a PDF using pdfplumber (better layout) with pypdf fallback.
    """
    text = ""

    # Try pdfplumber first
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            pages_to_read = pdf.pages[:max_pages]
            for page in pages_to_read:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n\n"
        if text.strip():
            return text
    except Exception as e:
        log(f"  [yellow]pdfplumber failed ({e}), trying pypdf...[/yellow]")

    # Fallback to pypdf
    try:
        from pypdf import PdfReader
        reader = PdfReader(pdf_path)
        for page in reader.pages[:max_pages]:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n\n"
    except Exception as e:
        log(f"  [red]pypdf also failed: {e}[/red]")

    return text


# ─────────────────────────────────────────────────────────────────────────────
# Step 3: Analyze each report with Claude
# ─────────────────────────────────────────────────────────────────────────────
EXTRACTION_PROMPT = """You are a senior equity analyst assistant. Analyze this broker research report and extract key information in JSON format.

Report Details:
- Quarter: {quarter}
- Ticker: {ticker}
- Broker: {broker_name}
- Filename: {filename}

Report Text:
{report_text}

Extract and return ONLY a valid JSON object (no markdown, no explanation) with these fields:

{{
  "rating": "BUY/ADD/HOLD/REDUCE/SELL or null",
  "target_price": "numeric value or null",
  "target_price_currency": "INR or null",
  "target_price_upside_pct": "numeric % upside/downside from CMP or null",
  "cmp": "current market price at time of report or null",
  "revenue_actual": "actual revenue for the quarter (in Cr or Mn as reported) or null",
  "revenue_estimate": "estimated/consensus revenue or null",
  "revenue_beat_miss": "BEAT/MISS/IN-LINE or null",
  "ebitda_actual": "actual EBITDA or null",
  "ebitda_margin_actual": "EBITDA margin % or null",
  "pat_actual": "Profit After Tax actual or null",
  "pat_estimate": "PAT estimate or null",
  "pat_beat_miss": "BEAT/MISS/IN-LINE or null",
  "eps_actual": "EPS actual or null",
  "eps_fy_estimate": "full year EPS estimate (current FY) or null",
  "eps_next_fy_estimate": "next FY EPS estimate or null",
  "management_guidance": "1-3 sentences on what management said about future outlook",
  "key_positives": ["list of 2-4 key positive points from the report"],
  "key_negatives": ["list of 2-4 key concerns or risks"],
  "analyst_view": "1-2 sentences summarizing the analyst's overall view",
  "key_themes": ["2-3 main themes or catalysts mentioned"],
  "previous_target_price": "previous target price if mentioned or null",
  "rating_change": "UPGRADED/DOWNGRADED/MAINTAINED or null"
}}

Use null for any field you cannot find. Be precise with numbers."""


def analyze_report_with_claude(report: dict, text: str, client) -> dict:
    """Sends report text to Claude API and gets structured analysis."""
    prompt = EXTRACTION_PROMPT.format(
        quarter=report["quarter"],
        ticker=report["ticker"],
        broker_name=report["broker_name"],
        filename=report["filename"],
        report_text=text[:12000],  # Limit to avoid token overflow
    )

    try:
        message = client.messages.create(
            model=CONFIG["model"],
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )
        response_text = message.content[0].text.strip()

        # Clean up JSON if needed
        if response_text.startswith("```"):
            response_text = re.sub(r"```json?\n?", "", response_text)
            response_text = response_text.replace("```", "").strip()

        data = json.loads(response_text)
        data["_extraction_success"] = True
        return data

    except json.JSONDecodeError as e:
        log(f"  [yellow]JSON parse error: {e}[/yellow]")
        return {"_extraction_success": False, "_error": str(e)}
    except Exception as e:
        log(f"  [red]API error: {e}[/red]")
        return {"_extraction_success": False, "_error": str(e)}


# ─────────────────────────────────────────────────────────────────────────────
# Step 4: Generate longitudinal synthesis
# ─────────────────────────────────────────────────────────────────────────────
SYNTHESIS_PROMPT = """You are a senior equity research analyst. Below is structured data extracted from {num_reports} broker research reports for {ticker} spanning {quarters} quarters.

Analyze this longitudinal data and write a comprehensive narrative insight report covering:

1. **Financial Trajectory**: How has revenue, PAT, EPS, and margins trended? Were there quarters of beats or misses?

2. **Analyst Confidence Over Time**: How have target prices changed? Have ratings been upgraded or downgraded? What does this signal?

3. **Management Credibility**: What guidance did management give, and did subsequent quarters validate or contradict it?

4. **Key Themes Evolution**: Which themes dominated early reports vs recent ones? What changed in the business story?

5. **Red Flags & Green Flags**: Any consistent concerns that never got resolved? Any positives that kept delivering?

6. **Current Analyst Consensus**: Based on the most recent reports, what is the overall view?

7. **Investment Verdict**: Based purely on this 8-12 quarter track record, what's your honest 2-3 sentence assessment?

Write in clear, professional language suitable for a retail investor who follows the stock.

Report Data:
{report_data}
"""


def generate_synthesis(ticker: str, analyzed_reports: list, client) -> str:
    """Generates the final longitudinal insight narrative."""
    # Prepare a clean summary of all reports for the synthesis prompt
    report_summaries = []
    quarters_seen = set()

    for r in analyzed_reports:
        q = r["quarter"]
        quarters_seen.add(q)
        summary = {
            "quarter": q,
            "broker": r["broker_name"],
            "data": {k: v for k, v in r["analysis"].items() if not k.startswith("_")}
        }
        report_summaries.append(summary)

    data_str = json.dumps(report_summaries, indent=2)

    prompt = SYNTHESIS_PROMPT.format(
        num_reports=len(analyzed_reports),
        ticker=ticker,
        quarters=", ".join(sorted(quarters_seen)),
        report_data=data_str[:15000],
    )

    try:
        message = client.messages.create(
            model=CONFIG["model"],
            max_tokens=3000,
            messages=[{"role": "user", "content": prompt}]
        )
        return message.content[0].text
    except Exception as e:
        return f"Error generating synthesis: {e}"


# ─────────────────────────────────────────────────────────────────────────────
# Step 5: Format and save output
# ─────────────────────────────────────────────────────────────────────────────
def format_html_report(ticker: str, analyzed_reports: list, synthesis: str) -> str:
    """Creates a beautiful HTML report."""
    rows = ""
    for r in analyzed_reports:
        a = r["analysis"]
        rating_color = {
            "BUY": "#16a34a", "ADD": "#65a30d", "HOLD": "#d97706",
            "REDUCE": "#ea580c", "SELL": "#dc2626"
        }.get(str(a.get("rating", "")).upper(), "#6b7280")

        change_badge = ""
        if a.get("rating_change"):
            colors = {"UPGRADED": "#16a34a", "DOWNGRADED": "#dc2626", "MAINTAINED": "#6b7280"}
            c = colors.get(a["rating_change"], "#6b7280")
            change_badge = f'<span style="color:{c};font-size:0.75rem;">▶ {a["rating_change"]}</span>'

        rows += f"""
        <tr>
            <td><strong>{r['quarter']}</strong></td>
            <td>{r['broker_name']}</td>
            <td style="color:{rating_color};font-weight:bold;">{a.get('rating','—')}</td>
            <td>₹{a.get('target_price','—')} {change_badge}</td>
            <td>₹{a.get('cmp','—')}</td>
            <td>{a.get('target_price_upside_pct','—')}%</td>
            <td>{a.get('pat_actual','—')}</td>
            <td><span class="beat-miss {str(a.get('pat_beat_miss','')).lower()}">{a.get('pat_beat_miss','—')}</span></td>
            <td>{a.get('eps_fy_estimate','—')}</td>
        </tr>"""

    synthesis_html = synthesis.replace("\n", "<br>").replace("**", "<strong>").replace("**", "</strong>")
    # Proper bold formatting
    synthesis_html = re.sub(r'\*\*(.*?)\*\*', r'<strong>\1</strong>', synthesis)
    synthesis_html = synthesis_html.replace("\n\n", "</p><p>").replace("\n", "<br>")

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{ticker} — Research Insights</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
         background: #f8fafc; color: #1e293b; line-height: 1.6; }}
  .header {{ background: linear-gradient(135deg, #1e3a5f 0%, #2d6a9f 100%);
             color: white; padding: 2.5rem 2rem; }}
  .header h1 {{ font-size: 2rem; font-weight: 700; }}
  .header p {{ opacity: 0.8; margin-top: 0.25rem; font-size: 0.95rem; }}
  .container {{ max-width: 1200px; margin: 0 auto; padding: 2rem; }}
  .card {{ background: white; border-radius: 12px; padding: 1.75rem;
           box-shadow: 0 1px 3px rgba(0,0,0,0.1); margin-bottom: 1.5rem; }}
  h2 {{ font-size: 1.2rem; font-weight: 600; color: #1e3a5f;
        margin-bottom: 1rem; padding-bottom: 0.5rem;
        border-bottom: 2px solid #e2e8f0; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.875rem; }}
  th {{ background: #f1f5f9; padding: 0.6rem 0.8rem; text-align: left;
        font-weight: 600; color: #475569; white-space: nowrap; }}
  td {{ padding: 0.6rem 0.8rem; border-bottom: 1px solid #e2e8f0; vertical-align: middle; }}
  tr:hover td {{ background: #f8fafc; }}
  .beat-miss {{ padding: 2px 8px; border-radius: 999px; font-size: 0.75rem; font-weight: 600; }}
  .beat {{ background: #dcfce7; color: #166534; }}
  .miss {{ background: #fee2e2; color: #991b1b; }}
  .in-line {{ background: #fef9c3; color: #713f12; }}
  .synthesis {{ font-size: 0.95rem; line-height: 1.8; color: #334155; }}
  .synthesis p {{ margin-bottom: 1rem; }}
  .synthesis strong {{ color: #1e3a5f; }}
  .meta {{ color: #94a3b8; font-size: 0.8rem; text-align: right; margin-top: 2rem; }}
  .tag {{ display: inline-block; background: #e0f2fe; color: #0369a1;
          padding: 2px 8px; border-radius: 4px; font-size: 0.75rem;
          margin: 2px; }}
</style>
</head>
<body>
<div class="header">
  <h1>📊 {ticker} — Longitudinal Research Insights</h1>
  <p>Generated on {datetime.now().strftime('%d %B %Y, %H:%M')} · {len(analyzed_reports)} reports analyzed</p>
</div>
<div class="container">

  <div class="card">
    <h2>🔍 Analyst Insight Synthesis</h2>
    <div class="synthesis"><p>{synthesis_html}</p></div>
  </div>

  <div class="card">
    <h2>📋 Report-by-Report Summary</h2>
    <div style="overflow-x:auto;">
    <table>
      <thead>
        <tr>
          <th>Quarter</th><th>Broker</th><th>Rating</th>
          <th>Target Price</th><th>CMP</th><th>Upside</th>
          <th>PAT Actual</th><th>PAT vs Est</th><th>FY EPS Est</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>
    </div>
  </div>

  <div class="meta">Generated by Stock Research Analyzer · Powered by Claude AI</div>
</div>
</body>
</html>"""


def format_text_report(ticker: str, analyzed_reports: list, synthesis: str) -> str:
    """Creates a plain text report."""
    lines = [
        "=" * 70,
        f"  {ticker} — Longitudinal Research Insights",
        f"  Generated: {datetime.now().strftime('%d %B %Y, %H:%M')}",
        f"  Reports analyzed: {len(analyzed_reports)}",
        "=" * 70,
        "",
        "ANALYST SYNTHESIS",
        "-" * 70,
        synthesis,
        "",
        "REPORT SUMMARY TABLE",
        "-" * 70,
        f"{'Quarter':<12} {'Broker':<22} {'Rating':<8} {'Target':>8} {'CMP':>8} {'PAT':>10} {'Beat/Miss':<10}",
        "-" * 70,
    ]

    for r in analyzed_reports:
        a = r["analysis"]
        lines.append(
            f"{r['quarter']:<12} {r['broker_name']:<22} "
            f"{str(a.get('rating','—')):<8} "
            f"{'₹'+str(a.get('target_price','—')):>8} "
            f"{'₹'+str(a.get('cmp','—')):>8} "
            f"{str(a.get('pat_actual','—')):>10} "
            f"{str(a.get('pat_beat_miss','—')):<10}"
        )

    lines.append("")
    return "\n".join(lines)


# ─────────────────────────────────────────────────────────────────────────────
# Main Orchestrator
# ─────────────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser(
        description="Analyze stock research reports across quarters using Claude AI",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=textwrap.dedent("""
        Examples:
          python stock_research_analyzer.py --ticker RELIANCE
          python stock_research_analyzer.py --ticker HDFCBANK --quarters 8
          python stock_research_analyzer.py --ticker TITAN --output html
          python stock_research_analyzer.py --ticker INFY --quarters 12 --output both
        """)
    )
    parser.add_argument("--ticker", required=True, help="Stock ticker (e.g. RELIANCE, HDFCBANK)")
    parser.add_argument("--quarters", type=int, default=10, help="Number of past quarters to analyze (default: 10)")
    parser.add_argument("--output", choices=["html", "text", "both"], default="html",
                        help="Output format (default: html)")
    parser.add_argument("--folder", help="Override base folder path from CONFIG")
    parser.add_argument("--api-key", help="Override Anthropic API key from CONFIG")
    parser.add_argument("--no-synthesis", action="store_true",
                        help="Skip synthesis (just extract data, faster & cheaper)")
    args = parser.parse_args()

    ticker = args.ticker.upper()

    # Resolve folder
    base_folder = args.folder or CONFIG["base_folder"]
    output_folder = CONFIG["output_folder"] or base_folder

    # Resolve API key
    api_key = args.api_key or CONFIG["anthropic_api_key"] or os.environ.get("ANTHROPIC_API_KEY", "")
    if not api_key:
        log("""
[red]ERROR: No Anthropic API key found.[/red]

Please either:
  1. Edit CONFIG in this script: set "anthropic_api_key" = "sk-ant-..."
  2. Set environment variable: set ANTHROPIC_API_KEY=sk-ant-...
  3. Pass via command line: --api-key sk-ant-...

Get your FREE API key at: https://console.anthropic.com
(Free tier gives ~$5 credit — enough for 50-100 analyses)
""")
        sys.exit(1)

    # Initialize Anthropic client
    try:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
    except ImportError:
        log("[red]ERROR: anthropic package not installed. Run: pip install anthropic[/red]")
        sys.exit(1)

    log(f"\n[bold cyan]═══ Stock Research Analyzer ═══[/bold cyan]")
    log(f"Ticker: [bold]{ticker}[/bold]  |  Quarters: {args.quarters}  |  Output: {args.output}")

    # Step 1: Find all reports
    log(f"\n[bold]Step 1: Scanning folder for {ticker} reports...[/bold]")
    reports = discover_reports(base_folder, ticker, args.quarters)

    if not reports:
        log(f"\n[red]No reports found for ticker '{ticker}' in {base_folder}[/red]")
        log("Check that your folder path and file naming conventions are correct.")
        sys.exit(1)

    log(f"\n[green]Found {len(reports)} total report files for {ticker}[/green]")

    # Step 2 & 3: Extract text and analyze each report
    log(f"\n[bold]Step 2: Extracting and analyzing reports...[/bold]")
    analyzed_reports = []

    for report in reports:
        log(f"\n  Processing: [cyan]{report['filename']}[/cyan]")

        # Extract text
        log(f"    → Extracting PDF text...")
        text = extract_pdf_text(report["path"], CONFIG["max_pages_per_pdf"])

        if not text.strip():
            log(f"    [yellow]⚠ No text extracted (scanned/image PDF?)[/yellow]")
            report["analysis"] = {"_extraction_success": False, "_error": "No text extracted"}
            analyzed_reports.append(report)
            continue

        word_count = len(text.split())
        log(f"    → Extracted ~{word_count} words")

        # Analyze with Claude
        log(f"    → Analyzing with Claude...")
        analysis = analyze_report_with_claude(report, text, client)

        if analysis.get("_extraction_success"):
            log(f"    [green]✓ Rating: {analysis.get('rating','?')} | "
                f"Target: ₹{analysis.get('target_price','?')} | "
                f"PAT: {analysis.get('pat_beat_miss','?')}[/green]")
        else:
            log(f"    [red]✗ Analysis failed: {analysis.get('_error','unknown')}[/red]")

        report["analysis"] = analysis
        analyzed_reports.append(report)

    # Filter to successful analyses for synthesis
    successful = [r for r in analyzed_reports if r["analysis"].get("_extraction_success")]
    log(f"\n[green]{len(successful)}/{len(analyzed_reports)} reports successfully analyzed[/green]")

    # Step 4: Generate synthesis
    synthesis = ""
    if not args.no_synthesis and successful:
        log(f"\n[bold]Step 3: Generating longitudinal synthesis...[/bold]")
        synthesis = generate_synthesis(ticker, successful, client)
        log("[green]✓ Synthesis complete[/green]")
    elif args.no_synthesis:
        synthesis = "Synthesis skipped (--no-synthesis flag used)."
    else:
        synthesis = "Insufficient data for synthesis."

    # Step 5: Save output
    log(f"\n[bold]Step 4: Saving reports...[/bold]")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    saved_files = []

    if args.output in ("html", "both"):
        html_content = format_html_report(ticker, analyzed_reports, synthesis)
        html_file = output_path / f"{ticker}_insights_{timestamp}.html"
        html_file.write_text(html_content, encoding="utf-8")
        saved_files.append(str(html_file))
        log(f"  [green]✓ HTML report: {html_file}[/green]")

    if args.output in ("text", "both"):
        text_content = format_text_report(ticker, analyzed_reports, synthesis)
        txt_file = output_path / f"{ticker}_insights_{timestamp}.txt"
        txt_file.write_text(text_content, encoding="utf-8")
        saved_files.append(str(txt_file))
        log(f"  [green]✓ Text report: {txt_file}[/green]")

    # Also save raw JSON data for future use
    json_file = output_path / f"{ticker}_insights_{timestamp}.json"
    json_file.write_text(
        json.dumps({"ticker": ticker, "generated": timestamp, "reports": analyzed_reports}, indent=2, default=str),
        encoding="utf-8"
    )
    log(f"  [green]✓ Raw data: {json_file}[/green]")

    log(f"\n[bold green]═══ Done! ═══[/bold green]")
    log(f"Open the HTML file in your browser for the full insight report.\n")


if __name__ == "__main__":
    main()
